# Aspect-Specific Augmentation Contrastive Learning

Learn a **fixed-partitioned** 64-dim embedding where each 16-dim chunk is
structurally tied to a semantic group:

| chunk | dims | semantic |
|---|---|---|
| `balance`     | `[ 0:16]` | balance-block features |
| `payment`     | `[16:32]` | payment-block features |
| `delinquency` | `[32:48]` | delinquency-block features |
| `others`      | `[48:64]` | everything else |

Input: 76 numeric features × 24 months. Each raw feature is mapped to one of
the four semantic groups (19 features per group here).

## How aspect-specific augmentation works

For each aspect `g`, build a **g-preserving view**: features outside `g` are
perturbed (we shuffle them across the batch by default), features inside `g`
are intact. Run the encoder on the anchor view AND on each g-preserving
view, then apply NT-Xent on **chunk g** between the (anchor, g-preserving)
pairs:

- **Positives** = same customer's chunk_g across the two views. Positives
  differ only in *non-g* features, so the contrastive force makes chunk_g
  invariant to them.
- **Negatives** = other customers in the batch. Negatives differ in g's own
  features, so chunk_g must remain discriminative on g-content.

Both forces together pin chunk_g to aspect g's features by construction — no
cluster labels needed.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch

from ts_embed.data import aspect_preserving_view
from ts_embed.model import TSEncoder, TSEncoderConfig
from ts_embed.loss import AspectSpec, AspectAugContrastiveLoss

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Configuration: 76 features, 4 semantic groups, 16-dim chunks

In [ ]:
N, T, F_NUM = 4000, 24, 76
D_MODEL = 64
ASPECTS = ['balance', 'payment', 'delinquency', 'others']
ASPECT_TO_ID = {n: i for i, n in enumerate(ASPECTS)}
N_ASPECTS = len(ASPECTS)
DIM_PER_CHUNK = D_MODEL // N_ASPECTS                  # 64 / 4 = 16
N_PER_ASPECT  = F_NUM // N_ASPECTS                    # 76 / 4 = 19
K_PER_ASPECT  = 3                                     # latent groups per aspect (for sanity-check only)

print(f'd_model={D_MODEL}  chunk dim={DIM_PER_CHUNK}')
print(f'F_NUM={F_NUM}  features per aspect={N_PER_ASPECT}')
print(f'aspects={ASPECTS}')

## 2. Feature -> aspect mapping

Each of the 76 numeric features is tagged with its semantic group via a long
tensor of shape `(76,)`. Here we use contiguous blocks of 19 features per
group; in production this comes from your data dictionary.

In [ ]:
# feature_group[i] = aspect id of feature i.
feature_group_np = np.repeat(np.arange(N_ASPECTS), N_PER_ASPECT).astype(np.int64)
feature_group = torch.from_numpy(feature_group_np).to(device)

for a_name, a_id in ASPECT_TO_ID.items():
    idx = np.flatnonzero(feature_group_np == a_id)
    print(f'  {a_name:12s} (id={a_id}): features {idx.min()}..{idx.max()} '
          f'({len(idx)} features)')

## 3. Synthetic data with aspect-specific latent groups

Four **independent** latent group factors (one per aspect) drive their own
feature block: balance latent only drives balance features, etc. Independence
is what lets us later check **specialization** — a well-trained `balance`
chunk should predict the balance latent group well but the others near
chance.

In [ ]:
rng = np.random.default_rng(0)
latent_groups_np = {a: rng.integers(0, K_PER_ASPECT, N) for a in ASPECTS}

numeric = (0.3 * rng.standard_normal((N, T, F_NUM))).astype(np.float32)
t_axis = np.linspace(0, 1, T, dtype=np.float32)[None, :, None]

for a_id, a_name in enumerate(ASPECTS):
    sl = slice(a_id * N_PER_ASPECT, (a_id + 1) * N_PER_ASPECT)
    groups = latent_groups_np[a_name]
    # Per-group template (mean signature) and per-group trend slope.
    templates = rng.standard_normal((K_PER_ASPECT, N_PER_ASPECT)).astype(np.float32) * 1.4
    trends    = rng.standard_normal(K_PER_ASPECT).astype(np.float32) * 0.7
    for g in range(K_PER_ASPECT):
        m = groups == g
        n_g = int(m.sum())
        base = templates[g][None, None, :]
        numeric[m, :, sl] += base + trends[g] * t_axis
        numeric[m, :, sl] += 0.25 * rng.standard_normal((n_g, T, N_PER_ASPECT)).astype(np.float32)

# 10% missing-at-random, then 'impute' with feature mean (matches the encoder's
# missing-data pathway).
missing = (rng.random((N, T, F_NUM)) < 0.1).astype(np.float32)
feat_mean = numeric.mean((0, 1), keepdims=True)
numeric = np.where(missing == 1.0, feat_mean, numeric).astype(np.float32)

numeric_t = torch.from_numpy(numeric)                       # (N, T, F)
missing_t = torch.from_numpy(missing)                       # (N, T, F)
print('numeric_t:', tuple(numeric_t.shape), 'missing_rate:', float(missing_t.mean()))
for a in ASPECTS:
    print(f'  {a:12s} latent counts: {np.bincount(latent_groups_np[a])}')

## 4. Demo: what does `aspect_preserving_view` do?

Take a small batch and show that calling `aspect_preserving_view(..., aspect_id=balance)` leaves the
balance feature block intact and perturbs everything else. Default mode is
`'shuffle'` (each non-balance feature is replaced with another customer's
value — preserves distributions, breaks customer↔feature linkage).

In [ ]:
demo_idx = torch.arange(8)
demo_num = numeric_t[demo_idx].to(device)
demo_mis = missing_t[demo_idx].to(device)

balance_id = ASPECT_TO_ID['balance']
aug_num, aug_mis, _ = aspect_preserving_view(
    demo_num, demo_mis, None, feature_group, aspect_id=balance_id, mode='shuffle')

balance_slice = slice(0, N_PER_ASPECT)
payment_slice = slice(N_PER_ASPECT, 2 * N_PER_ASPECT)
diff_balance = (aug_num[:, :, balance_slice] - demo_num[:, :, balance_slice]).abs().max().item()
diff_payment = (aug_num[:, :, payment_slice] - demo_num[:, :, payment_slice]).abs().max().item()
print(f'preserving aspect=balance with mode="shuffle":')
print(f'  max change in BALANCE feature block: {diff_balance:.4f}  (should be 0)')
print(f'  max change in PAYMENT feature block: {diff_payment:.4f}  (should be > 0)')

## 5. Encoder + AspectAugContrastiveLoss

`d_model=64` partitioned into four 16-dim chunks. `cat_cardinalities=()` for
this demo (no categorical features). `AspectAugContrastiveLoss` carries one
small projection head per chunk.

In [ ]:
cfg = TSEncoderConfig(
    n_numeric=F_NUM,
    cat_cardinalities=(),               # no categorical for this demo
    seq_len=T,
    d_model=D_MODEL,
    n_layers=3,
    n_heads=4,
    proj_dim=64,                        # unused -- we use the per-chunk heads instead
)
encoder = TSEncoder(cfg).to(device)

# 4 chunks of 16 dims each, slicing the 64-dim encoder output.
specs = [
    AspectSpec(name=a, start=i * DIM_PER_CHUNK, end=(i + 1) * DIM_PER_CHUNK,
               proj_dim=32, proj_hidden=64, temperature=0.1, weight=1.0)
    for i, a in enumerate(ASPECTS)
]
aspect_loss = AspectAugContrastiveLoss(
    specs, decorr_weight=0.5, var_weight=0.5, var_gamma=1.0,
).to(device)

params = list(encoder.parameters()) + list(aspect_loss.parameters())
optim = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
print(f'encoder params: {sum(p.numel() for p in encoder.parameters())/1e6:.2f}M')
print(f'loss heads params: {sum(p.numel() for p in aspect_loss.parameters())/1e6:.2f}M')
for s in specs:
    print(f'  {s.name:12s} chunk: [{s.start:2d}, {s.end:2d})  proj_dim={s.proj_dim}')

## 6. Training loop

Each step we make **N+1 = 5** encoder forward passes per batch:
- 1 on the anchor view
- 1 per aspect on the corresponding g-preserving view

For a small encoder this is fine; for production you'd stack the 5 views into
one `(5B, T, F)` batch and run the encoder once (see Next Steps).

Losses to watch:
- `c_<aspect>` -- per-chunk NT-Xent (should fall).
- `decorr` -- mean squared cross-chunk correlation (should fall toward 0).
- `var` -- variance hinge (stays near 0 if chunks are healthy).

In [ ]:
BATCH = 256
EPOCHS = 10
AUG_MODE = 'shuffle'    # 'mask' | 'shuffle' | 'noise'

history = []
for epoch in range(EPOCHS):
    encoder.train(); aspect_loss.train()
    perm = np.random.default_rng(epoch).permutation(N)
    n_batches = N // BATCH
    epoch_totals = {}
    for b in range(n_batches):
        idx = torch.from_numpy(perm[b * BATCH:(b + 1) * BATCH]).long()
        num = numeric_t[idx].to(device)
        mis = missing_t[idx].to(device)

        emb_anchor = encoder(num, mis, None, None)
        emb_per_aspect = {}
        for a_name, a_id in ASPECT_TO_ID.items():
            n_g, m_g, _ = aspect_preserving_view(
                num, mis, None, feature_group, aspect_id=a_id, mode=AUG_MODE)
            emb_per_aspect[a_name] = encoder(n_g, m_g, None, None)

        out = aspect_loss(emb_anchor, emb_per_aspect)
        loss = out['loss']

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optim.step()
        for k, v in out.items():
            epoch_totals[k] = epoch_totals.get(k, 0.0) + float(v)

    epoch_avg = {k: v / max(n_batches, 1) for k, v in epoch_totals.items()}
    history.append(epoch_avg)
    print(f"epoch {epoch:2d}  loss={epoch_avg['loss']:.3f}  "
          f"c_balance={epoch_avg['c_balance']:.3f} c_payment={epoch_avg['c_payment']:.3f} "
          f"c_deliq={epoch_avg['c_delinquency']:.3f} c_others={epoch_avg['c_others']:.3f}  "
          f"decorr={epoch_avg['decorr']:.4f}  var={epoch_avg['var']:.3f}")

## 7. Specialization check

For each (chunk, latent-aspect) pair, fit a cosine-kNN classifier on the
chunk's embeddings and measure accuracy at predicting that latent aspect's
group. A well-specialized embedding has a **diagonal-heavy** 4x4 matrix:
each chunk recovers its own latent group, but not the others'.

In [ ]:
@torch.no_grad()
def embed_all():
    encoder.eval()
    Z = []
    for s in range(0, N, 512):
        e = min(s + 512, N)
        num = numeric_t[s:e].to(device)
        mis = missing_t[s:e].to(device)
        Z.append(encoder(num, mis, None, None).float().cpu().numpy())
    return np.concatenate(Z)

def knn_acc(X, y, k=15, seed=0):
    rng_local = np.random.default_rng(seed)
    perm = rng_local.permutation(len(X))
    cut = int(0.7 * len(X)); tr, te = perm[:cut], perm[cut:]
    Xt = X[tr] / (np.linalg.norm(X[tr], axis=1, keepdims=True) + 1e-8)
    Xe = X[te] / (np.linalg.norm(X[te], axis=1, keepdims=True) + 1e-8)
    nn = np.argpartition(-(Xe @ Xt.T), k, axis=1)[:, :k]
    pred = np.array([np.bincount(r).argmax() for r in y[tr][nn]])
    return float((pred == y[te]).mean())

Z = embed_all()                          # (N, 64)
chunks_emb = {a: Z[:, i * DIM_PER_CHUNK:(i + 1) * DIM_PER_CHUNK]
              for i, a in enumerate(ASPECTS)}

print('kNN accuracy   rows=chunk, cols=latent group of aspect   '
      f'chance={1.0 / K_PER_ASPECT:.2f}')
print('              ' + '  '.join(f'{a:>12s}' for a in ASPECTS))
for chunk_name in ASPECTS:
    row = [knn_acc(chunks_emb[chunk_name], latent_groups_np[t])
           for t in ASPECTS]
    print(f'{chunk_name:12s}  ' + '  '.join(f'{v:12.3f}' for v in row))

# Collapse guard per chunk.
for a in ASPECTS:
    s = chunks_emb[a].std(0)
    assert s.mean() > 1e-2, f'COLLAPSE in {a} chunk'
    print(f'  {a:12s} per-dim std: mean={s.mean():.3f} min={s.min():.3f}')

# Cross-chunk correlation matrix (disentanglement check).
Zn = {a: (chunks_emb[a] - chunks_emb[a].mean(0)) /
         (chunks_emb[a].std(0) + 1e-8) for a in ASPECTS}
print('\nmean |cross-correlation| between chunks (off-diagonal should be ~0):')
print('              ' + '  '.join(f'{a:>12s}' for a in ASPECTS))
for a in ASPECTS:
    row = [float(np.abs(Zn[a].T @ Zn[b] / N).mean()) for b in ASPECTS]
    print(f'{a:12s}  ' + '  '.join(f'{v:12.3f}' for v in row))

## Next steps

- **Speed up training**: stack the 5 views into a single `(5B, T, F)` batch
  and run the encoder *once* per step. Memory cost up to ~5x; throughput
  improves because there's less per-call CUDA overhead.
- **Augmentation mode**: try `'mask'` (strongest — completely hides non-aspect
  features) or `'noise'` (weakest — preserves marginals). `'shuffle'` is
  a good default because it keeps the missingness distribution stable so the
  encoder can't trivially detect the augmentation type from the missing
  pattern.
- **Add aspect-aware categorical handling**: if you have categoricals that
  belong to specific aspects (e.g. revolve flag → balance/payment), extend
  `aspect_preserving_view` to also perturb non-aspect categoricals.
- **Combine with target supervision**: add a `SupConLoss` term on the full
  embedding using your downstream target label. The aspect chunks remain
  specialized; the overall embedding picks up target alignment.
- **Mix augmentations**: add light time-span masking (`TimeFeatureMasker`)
  on top of the aspect-preserving views for extra robustness.
- **Production pretrain**: replace the in-memory data with the memmap
  `TimeSeriesDataset` and reuse the DDP launcher pattern in
  `scripts/train_ddp.py`. The aspect-preserving augmentation is cheap and
  drops in alongside the existing collator.